## Demo: Part 1 - Benchmarking

In [ ]:
using QuadGK              # numerical ∫
using BenchmarkTools

: 

In [ ]:
# 1. analytic overlap of two s‑type primitive Gaussians 
# (centers coincide)
# S_G  =  (π / (α + β))^(3/2)
gauss_overlap(α, β) = (π / (α + β))^(3/2)

In [ ]:
# 2. numerical overlap of two Slater 1s functions (centres coincide)
#    Normalised STO: χ(r) = (ζ^3/π)^{1/2} * exp(-ζ r)
function slater_overlap(ζ1, ζ2)
    pref = (ζ1^3 * ζ2^3)^(1/2) / π
    integrand(r) = 4π * pref * exp(-(ζ1 + ζ2) * r) * r^2
    QuadGK.quadgk(integrand, 0.0, Inf; atol = 1e-12, rtol = 1e-12)[1]
end

In [ ]:
# --- demo parameters ---------------------------------------------------------
α  = 1.24     # Gaussian exponent
β  = 0.75
ζ1 = 1.24     # Slater effective charges
ζ2 = 0.75

In [ ]:
println("Single‑shot results:")
println("  Gaussian overlap  = ", gauss_overlap(α, β))
println("  Slater  overlap   = ", slater_overlap(ζ1, ζ2))

In [ ]:
println("\nTiming (100 000 evaluations each):")
@btime gauss_overlap($α, $β)        setup=() evals=100000
@btime slater_overlap($ζ1, $ζ2)     setup=() evals=100000

## Demo: Part 2 - Test

In [ ]:
using BasisSets
using OohataHuzinaga

In [11]:
mol = molecule("/Users/leticiamadureira/Projects/BasisSets.jl/test/data/hydrogen/hydrogen.xyz")

Molecule(["H", "H"], [0.0 0.0 0.0; 0.0 0.0 0.74], [1, 1])

In [12]:
mol.atoms

2-element Vector{String}:
 "H"
 "H"

In [33]:
mol = molecule("/Users/leticiamadureira/Projects/BasisSets.jl/test/data/hydrogen/hydrogen.xyz")

Molecule(["H", "H"], [0.0 0.0 -0.00129840845766; 0.0 0.0 0.74129840845766], [1, 1])

In [ ]:
shell_table = Dict(
           "H" => [(1, 0, 3),(2, 1, 1)],  
           "He" => [(1, 0, 3),(2, 1, 1)],
           "Li" => [(1, 0, 3),(2, 1, 1)],
           "Be" => [(1, 0, 3),(2, 1, 1)]
                   )


Dict{String, Vector{Tuple{Int64, Int64, Int64}}} with 1 entry:
  "H" => [(1, 0, 3)]

In [35]:
basis = BasisSets.optimizebasis(mol, shell_table)

Optimizing basis for atom: H
Optimizing for n=1, l=0, k=3
[0.24999999999999997 0.7999999999999999 2.56] ℓ = 0
 m = 0
 n = 0
Optimizing basis for atom: H
Optimizing for n=1, l=0, k=3
[0.24999999999999997 0.7999999999999999 2.56] ℓ = 0
 m = 0
 n = 0


2-element Vector{GaussianBasisSet}:
 GaussianBasisSet([0.0 0.0 -0.00129840845766], [0.24999999999999997 0.7999999999999999 2.56], [0.8000760295486874 0.040527319782266 0.8953541965717289], [0.2519794355383807 0.602875426920206 1.442414455797365], 3, 0, 0, 0)
 GaussianBasisSet([0.0 0.0 0.74129840845766], [0.24999999999999997 0.7999999999999999 2.56], [0.8000760295486874 0.040527319782266 0.8953541965717289], [0.2519794355383807 0.602875426920206 1.442414455797365], 3, 0, 0, 0)

In [36]:
basis[1]

GaussianBasisSet([0.0 0.0 -0.00129840845766], [0.24999999999999997 0.7999999999999999 2.56], [0.8000760295486874 0.040527319782266 0.8953541965717289], [0.2519794355383807 0.602875426920206 1.442414455797365], 3, 0, 0, 0)

In [37]:
res=OohataHuzinaga.rhf(basis,mol)


Overlap is done!
Kinetic is done!
Attraction is done!
Repulsion is done!
HCore is done!
Starting SCF iterations...
-1.5806966690624586
1.3466257560245267
-0.5877253240925875
1.3466257560245267


Results(-0.5877253240925886, -1.9343510801171153, 1.3466257560245267, [0.26376969057403465 0.26376969057403454; 0.26376969057403454 0.26376969057403443])

In [39]:
res.energy

-1.9343510801171153

# Water